
# Stage 8: temporal graph-learning notebook

This notebook keeps Stage 7 diffusion simulation separate from Stage 8 prediction. It uses the static directed Source -> Target diffusion graph and the processed monthly node features to predict next-month dominant sentiment labels without inventing timestamps or using future node information as model inputs.


In [1]:
import json
from copy import deepcopy
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from torch_geometric.nn import GCNConv

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

root = Path(r'C:\Users\Nikhil\Documents\sentiment_graph_diffusion')
processed = root / 'data' / 'processed'
results_dir = root / 'results'
(results_dir / 'tables').mkdir(parents=True, exist_ok=True)
(results_dir / 'figures').mkdir(parents=True, exist_ok=True)
(results_dir / 'models').mkdir(parents=True, exist_ok=True)


c:\Users\Nikhil\Documents\sentiment_graph_diffusion\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Nikhil\Documents\sentiment_graph_diffusion\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:

node_df = pd.read_csv(processed / 'diffusion_model_nodes.csv')
edge_df = pd.read_csv(processed / 'diffusion_model_edges.csv')
monthly_features = pd.read_csv(processed / 'diffusion_ready_monthly_node_features_full.csv')
monthly_summary = pd.read_csv(processed / 'diffusion_population_monthly_summary.csv')

node_order = node_df['node_id'].astype(str).tolist()
monthly_features = monthly_features[monthly_features['node_id'].isin(node_order)].copy()
months = sorted(monthly_features['month'].unique().tolist())
node_to_idx = {node: idx for idx, node in enumerate(node_order)}

assert len(node_order) == 17598
assert len(edge_df) == 29715
assert set(months) == {'2022-10', '2022-11', '2022-12', '2023-01', '2023-02', '2023-03'}
for month in months:
    count = monthly_features[monthly_features['month'] == month]['node_id'].nunique()
    assert count == len(node_order), (month, count, len(node_order))

source_idx = edge_df['Source'].map(node_to_idx)
target_idx = edge_df['Target'].map(node_to_idx)
assert source_idx.notna().all()
assert target_idx.notna().all()

edge_index = torch.tensor([source_idx.to_numpy(dtype=int), target_idx.to_numpy(dtype=int)], dtype=torch.long)
print('node_count', len(node_order))
print('edge_count', len(edge_df))
print('months', months)
print('tweet_count', monthly_summary['total_tweets'].sum())
print('edge_index shape', tuple(edge_index.shape))


node_count 17598
edge_count 29715
months ['2022-10', '2022-11', '2022-12', '2023-01', '2023-02', '2023-03']
tweet_count 37115
edge_index shape (2, 29715)


C:\Users\Nikhil\AppData\Local\Temp\ipykernel_32800\1135331581.py:23: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:255.)
  edge_index = torch.tensor([source_idx.to_numpy(dtype=int), target_idx.to_numpy(dtype=int)], dtype=torch.long)


In [3]:

feature_cols = [
    'tweet_count', 'avg_likes', 'avg_retweets', 'avg_polarity',
    'negative_count', 'positive_count', 'negative_ratio', 'positive_ratio'
]

feature_map = {}
label_by_month = {}
for month in months:
    month_df = monthly_features[monthly_features['month'] == month].copy().set_index('node_id').reindex(node_order).fillna(0.0)
    month_df[feature_cols] = month_df[feature_cols].astype(float)
    feature_map[month] = month_df[feature_cols].to_numpy(dtype=np.float32)
    positive = month_df['positive_ratio'].to_numpy(dtype=np.float32)
    negative = month_df['negative_ratio'].to_numpy(dtype=np.float32)
    label_by_month[month] = (positive >= negative).astype(np.int64)

assert all(np.isfinite(feature_map[m]).all() for m in months)
print('feature dims', {m: feature_map[m].shape for m in months})
print('positive share by month', {m: float((label_by_month[m] == 1).mean()) for m in months})


feature dims {'2022-10': (17598, 8), '2022-11': (17598, 8), '2022-12': (17598, 8), '2023-01': (17598, 8), '2023-02': (17598, 8), '2023-03': (17598, 8)}
positive share by month {'2022-10': 0.9122059324923286, '2022-11': 0.8783384475508581, '2022-12': 0.880327309921582, '2023-01': 0.8849869303329924, '2023-02': 0.8828844186839414, '2023-03': 0.8368564609614729}


In [4]:

class GCNBaseline(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        return self.head(x).squeeze(-1)


class EvolveGCNStyle(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.gcn1 = GCNConv(in_dim, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, hidden_dim)
        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=False)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, seq, edge_index):
        states = []
        for x in seq:
            x = x.to(torch.float32)
            x = F.relu(self.gcn1(x, edge_index))
            x = F.relu(self.gcn2(x, edge_index))
            states.append(x)
        seq_tensor = torch.stack(states, dim=0)
        out, _ = self.gru(seq_tensor.permute(1, 0, 2))
        final = out[:, -1, :]
        return self.head(final).squeeze(-1)


In [5]:
def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], average=None, zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    macro_f1 = float((f1[0] + f1[1]) / 2.0)
    weighted_f1 = float(precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], average='weighted', zero_division=0
    )[2])
    balanced_accuracy = float((recall[0] + recall[1]) / 2.0)
    pr_auc = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float('nan')
    return {
        'threshold': float(threshold),
        'accuracy': float(acc),
        'balanced_accuracy': balanced_accuracy,
        'negative_precision': float(precision[0]),
        'negative_recall': float(recall[0]),
        'negative_f1': float(f1[0]),
        'positive_precision': float(precision[1]),
        'positive_recall': float(recall[1]),
        'positive_f1': float(f1[1]),
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'pr_auc': float(pr_auc) if not np.isnan(pr_auc) else np.nan,
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


def select_threshold(y_true, y_prob):
    thresholds = np.unique(np.concatenate(([0.0, 0.5, 1.0], np.asarray(y_prob, dtype=float))))
    validation_metrics = [compute_binary_metrics(y_true, y_prob, threshold) for threshold in thresholds]
    best_index = max(
        range(len(validation_metrics)),
        key=lambda index: (validation_metrics[index]['macro_f1'], validation_metrics[index]['balanced_accuracy'], -thresholds[index]),
    )
    return float(thresholds[best_index]), validation_metrics[best_index]


all_pairs = [(months[i], months[i + 1]) for i in range(len(months) - 1)]
train_pairs = all_pairs[:-2]
val_pair = all_pairs[-2]
test_pair = all_pairs[-1]
print('train_pairs', train_pairs)
print('val_pair', val_pair)
print('test_pair', test_pair)


train_pairs [('2022-10', '2022-11'), ('2022-11', '2022-12'), ('2022-12', '2023-01')]
val_pair ('2023-01', '2023-02')
test_pair ('2023-02', '2023-03')


In [6]:
# Baseline training: predict the current month label from the current month features.
X_train = torch.tensor(feature_map[train_pairs[-1][0]], dtype=torch.float32)
X_val = torch.tensor(feature_map[val_pair[0]], dtype=torch.float32)
X_test = torch.tensor(feature_map[test_pair[0]], dtype=torch.float32)

y_train = torch.tensor(label_by_month[train_pairs[-1][1]], dtype=torch.float32)
y_val = torch.tensor(label_by_month[val_pair[1]], dtype=torch.float32)
y_test = torch.tensor(label_by_month[test_pair[1]], dtype=torch.float32)

base_model = GCNBaseline(in_dim=len(feature_cols), hidden_dim=32)
base_opt = torch.optim.Adam(base_model.parameters(), lr=1e-3, weight_decay=1e-4)
train_labels = y_train.detach().cpu().numpy().astype(int)
base_class_weights = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=train_labels)
base_sample_weights = torch.tensor(base_class_weights[train_labels], dtype=torch.float32)
base_criterion = nn.BCEWithLogitsLoss(weight=base_sample_weights)

best_state = None
best_val = float('inf')
patience = 8
wait = 0
for epoch in range(25):
    base_model.train()
    base_opt.zero_grad()
    logits = base_model(X_train, edge_index)
    loss = base_criterion(logits, y_train)
    loss.backward()
    base_opt.step()

    base_model.eval()
    with torch.no_grad():
        val_logits = base_model(X_val, edge_index)
        val_loss = base_criterion(val_logits, y_val)

    if float(val_loss.detach().cpu()) < best_val - 1e-6:
        best_val = float(val_loss.detach().cpu())
        best_state = deepcopy(base_model.state_dict())
        wait = 0
    else:
        wait += 1
    if wait >= patience:
        break

if best_state is not None:
    base_model.load_state_dict(best_state)

base_model.eval()
with torch.no_grad():
    base_val_prob = torch.sigmoid(base_model(X_val, edge_index)).cpu().numpy()
    base_test_prob = torch.sigmoid(base_model(X_test, edge_index)).cpu().numpy()

base_threshold, base_val_metrics = select_threshold(y_val.detach().cpu().numpy().astype(int), base_val_prob)
print('GCN validation threshold', base_threshold)
print('GCN validation metrics', base_val_metrics)

torch.save(base_model.state_dict(), results_dir / 'models' / 'gcn_baseline_imbalance_corrected.pt')


GCN validation threshold 0.48726534843444824
GCN validation metrics {'threshold': 0.48726534843444824, 'accuracy': 0.8254347084896011, 'balanced_accuracy': 0.5053402597490201, 'negative_precision': 0.13129102844638948, 'negative_recall': 0.08733624454148471, 'negative_f1': 0.1048951048951049, 'positive_precision': 0.8840820854132002, 'positive_recall': 0.9233442749565554, 'positive_f1': 0.9032867397053268, 'macro_f1': 0.5040909223002159, 'weighted_f1': 0.8097826392766493, 'pr_auc': 0.8841898573801354, 'tn': 180, 'fp': 1881, 'fn': 1191, 'tp': 14346}


In [7]:
# Temporal model training using the month history up to the current month.
train_hist = [
    [torch.tensor(feature_map[m], dtype=torch.float32) for m in months if months.index(m) <= months.index(in_month)]
    for in_month, _ in train_pairs
]
val_hist = [torch.tensor(feature_map[m], dtype=torch.float32) for m in months if months.index(m) <= months.index(val_pair[0])]
test_hist = [torch.tensor(feature_map[m], dtype=torch.float32) for m in months if months.index(m) <= months.index(test_pair[0])]

train_target = torch.tensor(label_by_month[train_pairs[-1][1]], dtype=torch.float32)
val_target = torch.tensor(label_by_month[val_pair[1]], dtype=torch.float32)
test_target = torch.tensor(label_by_month[test_pair[1]], dtype=torch.float32)

seq_model = EvolveGCNStyle(in_dim=len(feature_cols), hidden_dim=32)
seq_opt = torch.optim.Adam(seq_model.parameters(), lr=1e-3, weight_decay=1e-4)
train_labels = train_target.detach().cpu().numpy().astype(int)
seq_class_weights = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=train_labels)
seq_sample_weights = torch.tensor(seq_class_weights[train_labels], dtype=torch.float32)
seq_criterion = nn.BCEWithLogitsLoss(weight=seq_sample_weights)

best_state = None
best_val = float('inf')
patience = 8
wait = 0
for epoch in range(25):
    seq_model.train()
    seq_opt.zero_grad()
    logits = seq_model(train_hist[-1], edge_index)
    loss = seq_criterion(logits, train_target)
    loss.backward()
    seq_opt.step()

    seq_model.eval()
    with torch.no_grad():
        val_logits = seq_model(val_hist, edge_index)
        val_loss = seq_criterion(val_logits, val_target)

    if float(val_loss.detach().cpu()) < best_val - 1e-6:
        best_val = float(val_loss.detach().cpu())
        best_state = deepcopy(seq_model.state_dict())
        wait = 0
    else:
        wait += 1
    if wait >= patience:
        break

if best_state is not None:
    seq_model.load_state_dict(best_state)

seq_model.eval()
with torch.no_grad():
    seq_val_prob = torch.sigmoid(seq_model(val_hist, edge_index)).cpu().numpy()
    seq_test_prob = torch.sigmoid(seq_model(test_hist, edge_index)).cpu().numpy()

seq_threshold, seq_val_metrics = select_threshold(val_target.detach().cpu().numpy().astype(int), seq_val_prob)
print('EvolveGCN-style validation threshold', seq_threshold)
print('EvolveGCN-style validation metrics', seq_val_metrics)

torch.save(seq_model.state_dict(), results_dir / 'models' / 'evolvegcn_style_imbalance_corrected.pt')


EvolveGCN-style validation threshold 0.5187827944755554
EvolveGCN-style validation metrics {'threshold': 0.5187827944755554, 'accuracy': 0.7803727696329128, 'balanced_accuracy': 0.5073854785669631, 'negative_precision': 0.12819455894476506, 'negative_recall': 0.15089762251334304, 'negative_f1': 0.13862268776465345, 'positive_precision': 0.8846559451621407, 'positive_recall': 0.8638733346205831, 'positive_f1': 0.8741411312644501, 'macro_f1': 0.5063819095145518, 'weighted_f1': 0.7880004611852889, 'pr_auc': 0.8887093391271734, 'tn': 311, 'fp': 1750, 'fn': 2115, 'tp': 13422}


In [8]:
base_test_metrics = compute_binary_metrics(
    y_test.detach().cpu().numpy().astype(int), base_test_prob, threshold=base_threshold
)
seq_test_metrics = compute_binary_metrics(
    test_target.detach().cpu().numpy().astype(int), seq_test_prob, threshold=seq_threshold
)

rows = [
    {'model': 'GCN_baseline', 'split': 'validation', **base_val_metrics},
    {'model': 'GCN_baseline', 'split': 'test', **base_test_metrics},
    {'model': 'EvolveGCN_style', 'split': 'validation', **seq_val_metrics},
    {'model': 'EvolveGCN_style', 'split': 'test', **seq_test_metrics},
]
metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(results_dir / 'tables' / 'evolvegcn_metrics_imbalance_corrected.csv', index=False)
metrics_df.to_csv(results_dir / 'tables' / 'evolvegcn_model_comparison_imbalance_corrected.csv', index=False)
print(metrics_df)


             model       split  threshold  accuracy  balanced_accuracy  \
0     GCN_baseline  validation   0.487265  0.825435           0.505340   
1     GCN_baseline        test   0.487265  0.785828           0.497833   
2  EvolveGCN_style  validation   0.518783  0.780373           0.507385   
3  EvolveGCN_style        test   0.518783  0.746619           0.500765   

   negative_precision  negative_recall  negative_f1  positive_precision  \
0            0.131291         0.087336     0.104895            0.884082   
1            0.155146         0.070359     0.096813            0.836217   
2            0.128195         0.150898     0.138623            0.884656   
3            0.164696         0.135841     0.148883            0.837098   

   positive_recall  positive_f1  macro_f1  weighted_f1    pr_auc   tn    fp  \
0         0.923344     0.903287  0.504091     0.809783  0.884190  180  1881   
1         0.925307     0.878509  0.487661     0.750981  0.837082  202  2669   
2         0.8638

In [9]:
# Save predictions using thresholds selected on February validation only.
val_rows = []
for model_name, probs, threshold, target in [
    ('GCN_baseline', base_val_prob, base_threshold, y_val),
    ('EvolveGCN_style', seq_val_prob, seq_threshold, val_target),
]:
    for node_id, actual, prob in zip(node_order, target.detach().cpu().numpy().astype(int), probs):
        val_rows.append({
            'model': model_name,
            'month': val_pair[1],
            'node_id': node_id,
            'actual_label': int(actual),
            'predicted_probability': float(prob),
            'threshold': float(threshold),
            'predicted_label': int(prob >= threshold),
        })

test_rows = []
for model_name, probs, threshold, target in [
    ('GCN_baseline', base_test_prob, base_threshold, y_test),
    ('EvolveGCN_style', seq_test_prob, seq_threshold, test_target),
]:
    for node_id, actual, prob in zip(node_order, target.detach().cpu().numpy().astype(int), probs):
        test_rows.append({
            'model': model_name,
            'month': test_pair[1],
            'node_id': node_id,
            'actual_label': int(actual),
            'predicted_probability': float(prob),
            'threshold': float(threshold),
            'predicted_label': int(prob >= threshold),
        })

pred_df = pd.DataFrame(val_rows + test_rows)
pred_df.to_csv(results_dir / 'tables' / 'evolvegcn_monthly_predictions_imbalance_corrected.csv', index=False)
print(pred_df.head())


          model    month       node_id  actual_label  predicted_probability  \
0  GCN_baseline  2023-02       00012vv             1               0.532657   
1  GCN_baseline  2023-02      0002abhi             1               0.532657   
2  GCN_baseline  2023-02       0007_CJ             1               0.532657   
3  GCN_baseline  2023-02  001amitsingh             0               0.532657   
4  GCN_baseline  2023-02     002_akash             1               0.532657   

   threshold  predicted_label  
0   0.487265                1  
1   0.487265                1  
2   0.487265                1  
3   0.487265                1  
4   0.487265                1  


In [10]:
# Essential paper outputs: model comparison and held-out March confusion matrices.
comparison_df = metrics_df[metrics_df['split'] == 'test'].copy()
comparison_df.to_csv(results_dir / 'tables' / 'stage8_model_comparison_imbalance_corrected.csv', index=False)

march_targets = test_target.detach().cpu().numpy().astype(int)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for axis, model_name, probs, threshold in [
    (axes[0], 'GCN baseline', base_test_prob, base_threshold),
    (axes[1], 'EvolveGCN-style', seq_test_prob, seq_threshold),
]:
    cm = confusion_matrix(march_targets, (probs >= threshold).astype(int), labels=[0, 1])
    axis.imshow(cm, cmap='Blues')
    axis.set_title(f'{model_name}\nthreshold={threshold:.3f}')
    axis.set_xlabel('Predicted label')
    axis.set_ylabel('Actual label')
    axis.set_xticks([0, 1], ['Negative', 'Positive'])
    axis.set_yticks([0, 1], ['Negative', 'Positive'])
    for i in range(2):
        for j in range(2):
            axis.text(j, i, str(cm[i, j]), ha='center', va='center')
plt.suptitle('March 2023 held-out confusion matrices')
plt.tight_layout()
plt.savefig(results_dir / 'figures' / 'stage8_confusion_matrices_imbalance_corrected.png', dpi=200)
plt.close()
print('Saved corrected comparison table and confusion-matrix figure')


Saved corrected comparison table and confusion-matrix figure


In [11]:
base_f1 = base_test_metrics['macro_f1']
seq_f1 = seq_test_metrics['macro_f1']
interpretation = f'''## Interpretation

This corrected Stage 8 experiment preserves the original chronological setup: training uses the existing January-target training period, February 2023 is validation, and March 2023 is held out for one final test evaluation. Stage 7, raw datasets, and temporal feature construction were not changed.

- The previous all-positive result was a majority-class collapse caused by class imbalance: March 2023 contains 2871 negative samples (16.31%) and 14727 positive samples (83.69%), and the fixed 0.5 decision rule yielded zero negative recall and F1.
- Both the static GCN and EvolveGCN-style models use the same training principle: balanced class weights computed from training labels only.
- Each model selects its classification threshold on February validation using validation macro F1. That threshold is frozen before evaluating March; March labels are not used for training, threshold selection, or model selection.
- Corrected March 2023 metrics are saved in `stage8_model_comparison_imbalance_corrected.csv`; the full validation/test metrics are saved in `evolvegcn_metrics_imbalance_corrected.csv`.
- GCN baseline March metrics: {base_test_metrics}
- EvolveGCN-style March metrics: {seq_test_metrics}
- Any improvement claim must be based on these held-out March metrics; the imbalance correction alone is not evidence of improvement.
- The output remains predictive modeling of observed activity, not causal evidence that negative information spreads more quickly.
'''

Path(root / 'results' / 'tables' / 'evolvegcn_interpretation_imbalance_corrected.md').write_text(interpretation, encoding='utf-8')
print(interpretation)


## Interpretation

This corrected Stage 8 experiment preserves the original chronological setup: training uses the existing January-target training period, February 2023 is validation, and March 2023 is held out for one final test evaluation. Stage 7, raw datasets, and temporal feature construction were not changed.

- The previous all-positive result was a majority-class collapse caused by class imbalance: March 2023 contains 2871 negative samples (16.31%) and 14727 positive samples (83.69%), and the fixed 0.5 decision rule yielded zero negative recall and F1.
- Both the static GCN and EvolveGCN-style models use the same training principle: balanced class weights computed from training labels only.
- Each model selects its classification threshold on February validation using validation macro F1. That threshold is frozen before evaluating March; March labels are not used for training, threshold selection, or model selection.
- Corrected March 2023 metrics are saved in `stage8_model_co